In [11]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
URL = "https://raw.githubusercontent.com/ageron/handson-ml/refs/heads/master/datasets/housing/housing.csv"
df = pd.read_csv(URL)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [13]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=35)

X_train = train_set.drop("median_house_value", axis=1)
y = train_set["median_house_value"].copy()

X_num = X_train.drop("ocean_proximity", axis=1)

## Creating pipline

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self # nothing else to do
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder(add_bedrooms_per_room = True)),
    ('std_scaler', StandardScaler())
])

In [16]:
from sklearn.compose import ColumnTransformer

num_attribs = list(X_num)
cat_attribs = ['ocean_proximity']
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [17]:
X_prepared = full_pipeline.fit_transform(X_train)

In [18]:
X_prepared

array([[-1.25390838,  1.10757958, -1.79230424, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.31649014, -0.79863258, -1.2367069 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.65894633, -0.77989831,  0.66819829, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.80340671, -0.54103634,  0.27134305, ...,  0.        ,
         0.        ,  0.        ],
       [-1.12937357,  0.78909696, -0.52236745, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.62407658, -0.67217624,  0.58882724, ...,  0.        ,
         0.        ,  0.        ]], shape=(16512, 16))

## Linear Regression

In [39]:
from sklearn.linear_model import LinearRegression
LR_model = LinearRegression()

In [40]:
LR_model.fit(X_prepared, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [41]:
X_train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
1380,-122.09,38.00,6.0,10191.0,1882.0,4377.0,1789.0,5.2015,NEAR BAY
12294,-116.93,33.93,13.0,7804.0,1594.0,3297.0,1469.0,2.0549,INLAND
7387,-118.25,33.97,37.0,794.0,210.0,814.0,213.0,2.2917,<1H OCEAN
14454,-117.27,32.83,39.0,1877.0,426.0,805.0,409.0,3.8750,NEAR OCEAN
2927,-119.01,35.36,24.0,1941.0,484.0,1277.0,435.0,1.0560,INLAND
...,...,...,...,...,...,...,...,...,...
19391,-120.85,37.78,25.0,421.0,NaN,303.0,106.0,2.2679,INLAND
15393,-116.90,33.22,11.0,4132.0,773.0,2012.0,703.0,3.1906,<1H OCEAN
9143,-117.96,34.48,32.0,1896.0,342.0,806.0,299.0,4.5769,INLAND
17679,-121.84,37.32,22.0,3015.0,581.0,2491.0,530.0,4.3419,<1H OCEAN


In [58]:
test_data = X_train.sample(10)
test_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
18575,-121.77,36.91,8.0,2715.0,750.0,2580.0,718.0,2.8348,<1H OCEAN
17858,-121.87,37.46,43.0,91.0,12.0,58.0,16.0,15.0001,<1H OCEAN
18823,-121.62,41.78,40.0,3272.0,663.0,1467.0,553.0,1.7885,INLAND
7244,-118.10,34.00,32.0,2122.0,591.0,1929.0,539.0,2.7311,<1H OCEAN
16527,-121.21,37.81,12.0,3667.0,640.0,2173.0,652.0,5.0369,INLAND
6730,-118.14,34.11,52.0,2742.0,422.0,1153.0,414.0,8.1124,<1H OCEAN
11689,-117.99,33.87,16.0,1689.0,499.0,1260.0,453.0,3.1205,<1H OCEAN
855,-122.02,37.58,15.0,3052.0,760.0,2097.0,728.0,3.3617,NEAR BAY
16394,-121.25,38.03,29.0,2465.0,327.0,859.0,315.0,6.6605,INLAND
17714,-121.79,37.34,20.0,2018.0,328.0,1196.0,323.0,4.9318,<1H OCEAN


In [59]:
test_label = y.loc[test_data.index]
test_label

18575    162000.0
17858    500001.0
18823     43500.0
7244     169300.0
16527    163900.0
6730     500001.0
11689    174000.0
855      178100.0
16394    220700.0
17714    262400.0
Name: median_house_value, dtype: float64

In [60]:
test_data_prepared = full_pipeline.transform(test_data)
predicted_labels = LR_model.predict(test_data_prepared)

In [61]:
predicted_labels

array([172222.33466633, 665317.44077722,  10487.3262773 , 171734.80969008,
       181225.47969406, 402742.06871833, 187580.39476032, 202346.77360195,
       265370.67659109, 246665.76659321])

In [62]:
pd.DataFrame({
    "Prediction": predicted_labels,
    "Real": test_label, 
    "Difference": predicted_labels-test_label
})

,Prediction,Real,Difference
18575,172222.334666,162000.0,10222.334666
17858,665317.440777,500001.0,165316.440777
18823,10487.326277,43500.0,-33012.673723
7244,171734.809690,169300.0,2434.809690
16527,181225.479694,163900.0,17325.479694
6730,402742.068718,500001.0,-97258.931282
11689,187580.394760,174000.0,13580.394760
855,202346.773602,178100.0,24246.773602
16394,265370.676591,220700.0,44670.676591
17714,246665.766593,262400.0,-15734.233407


shouldʼve deleted >500k$ houses, ig

In [63]:
test_set

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
9288,-122.41,38.07,20.0,4536.0,708.0,1812.0,701.0,6.0433,435900.0,NEAR BAY
1878,-119.96,38.93,22.0,2731.0,632.0,1215.0,483.0,2.8300,110500.0,INLAND
20439,-118.80,34.27,12.0,3330.0,600.0,1577.0,584.0,4.6985,264100.0,<1H OCEAN
10957,-117.88,33.75,34.0,3004.0,673.0,5477.0,640.0,2.8342,187200.0,<1H OCEAN
10316,-117.80,33.85,16.0,4151.0,637.0,1558.0,604.0,5.8060,304900.0,<1H OCEAN
...,...,...,...,...,...,...,...,...,...,...
13044,-121.13,38.47,16.0,2574.0,441.0,1041.0,428.0,3.6645,203400.0,INLAND
11419,-117.96,33.69,20.0,3123.0,441.0,1319.0,432.0,6.0910,290400.0,<1H OCEAN
1804,-122.33,37.93,34.0,2326.0,471.0,1356.0,441.0,2.3475,90300.0,NEAR BAY
7136,-118.10,34.02,37.0,1022.0,232.0,653.0,238.0,3.0625,189400.0,<1H OCEAN


In [64]:
X_test = test_set.drop("median_house_value", axis=1)
X_test

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
9288,-122.41,38.07,20.0,4536.0,708.0,1812.0,701.0,6.0433,NEAR BAY
1878,-119.96,38.93,22.0,2731.0,632.0,1215.0,483.0,2.8300,INLAND
20439,-118.80,34.27,12.0,3330.0,600.0,1577.0,584.0,4.6985,<1H OCEAN
10957,-117.88,33.75,34.0,3004.0,673.0,5477.0,640.0,2.8342,<1H OCEAN
10316,-117.80,33.85,16.0,4151.0,637.0,1558.0,604.0,5.8060,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
13044,-121.13,38.47,16.0,2574.0,441.0,1041.0,428.0,3.6645,INLAND
11419,-117.96,33.69,20.0,3123.0,441.0,1319.0,432.0,6.0910,<1H OCEAN
1804,-122.33,37.93,34.0,2326.0,471.0,1356.0,441.0,2.3475,NEAR BAY
7136,-118.10,34.02,37.0,1022.0,232.0,653.0,238.0,3.0625,<1H OCEAN


In [65]:
y_test = test_set["median_house_value"].copy()
y_test

9288     435900.0
1878     110500.0
20439    264100.0
10957    187200.0
10316    304900.0
           ...   
13044    203400.0
11419    290400.0
1804      90300.0
7136     189400.0
16633    183900.0
Name: median_house_value, Length: 4128, dtype: float64

In [66]:
X_test_prepared = full_pipeline.transform(X_test)

In [67]:
y_predicted = LR_model.predict(X_test_prepared)

In [69]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_predicted)
mae

49810.06572460955

In [71]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_predicted)
np.sqrt(mse)

np.float64(69116.59360540153)

## Random forest

In [72]:
from sklearn.ensemble import RandomForestRegressor
RF_model = RandomForestRegressor()

In [73]:
RF_model.fit(X_prepared, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [75]:
y_predicted = RF_model.predict(X_test_prepared)

In [76]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_predicted)
np.sqrt(mse)

np.float64(50787.75360556537)

## Cross validation

In [77]:
X = train_set.drop("median_house_value", axis=1)
y = train_set["median_house_value"].copy()

X_prepared = full_pipeline.transform(X)

In [82]:
from sklearn.model_selection import cross_val_score
msescores = cross_val_score(LR_model, X_prepared, y, 
                         scoring="neg_mean_squared_error", cv=5)

In [83]:
def display_scores(scores):
    print("Scores: ", scores)
    print("Mean: ", scores.mean())
    print("Standard deviation: ", scores.std())

display_scores(np.sqrt(-msescores))

Scores:  [68925.38841132 69164.63385785 66161.87006568 71173.81099728
 66720.49462551]
Mean:  68429.23959152939
Standard deviation:  1809.9952765258754


In [84]:
scores = cross_val_score(RF_model, X_prepared, y, 
                         scoring="neg_mean_squared_error", cv=5)
LM_rmse_scores = np.sqrt(-scores)
display_scores(LM_rmse_scores)

Scores:  [50570.52534877 51859.11410541 49135.09450849 51441.25835038
 48610.39898355]
Mean:  50323.27825932004
Standard deviation:  1266.1388859418619


## Saving

In [85]:
import pickle
filename = "RF_model.pkl"
with open(filename, "wb") as file:
    pickle.dump(RF_model, file)

In [88]:
with open(filename, "rb") as file:
    model = pickle.load(file)

In [89]:
model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

joblib

In [90]:
import joblib
filename = "LR_model.jbl"
joblib.dump(LR_model, filename)

['LR_model.jbl']

In [91]:
model = joblib.load(filename)

In [92]:
scores = cross_val_score(model, X_prepared, y, 
                         scoring="neg_mean_squared_error", cv=5)
LM_rmse_scores = np.sqrt(-scores)
display_scores(LM_rmse_scores)

Scores:  [68925.38841132 69164.63385785 66161.87006568 71173.81099728
 66720.49462551]
Mean:  68429.23959152939
Standard deviation:  1809.9952765258754
